# The Hugging Face Datasets Library

Before we can train anything, we need data in a form the model can consume.
This notebook introduces the [`datasets`](https://huggingface.co/docs/datasets) library — the standard
tool for loading, inspecting, and streaming text datasets in the PyTorch ecosystem.

We use our own hosted copy of the tiny-Shakespeare corpus
(`transformingit/tiny-shakespeare` on the Hugging Face Hub) as the running example.

| Concept | What it means |
|---|---|
| **Hub** | Hugging Face's public registry of datasets and models |
| **`DatasetDict`** | A dict-like object holding one `Dataset` per split |
| **`Dataset`** | An Arrow-backed table — rows are examples, columns are features |
| **cache** | Downloaded data is stored in `~/.cache/huggingface/` — subsequent loads are instant |

## Section 1 — Loading from the Hub

`load_dataset` is the single entry point for everything on the Hub.
Pass it a dataset ID — `"owner/name"` — and it returns a `DatasetDict`.

In [ ]:
from datasets import load_dataset

raw = load_dataset("transformingit/tiny-shakespeare")
print(type(raw))
print(raw)

The result is a `DatasetDict` — a plain Python dict whose keys are split names.
Our dataset has one split (`"train"`) because Shakespeare is one continuous text
and we do our own 90/5/5 split in `protoform`.

In [ ]:
# Access a split by name — returns a Dataset (a table)
train_ds = raw["train"]
print(type(train_ds))
print(f"Rows:    {len(train_ds)}")
print(f"Columns: {train_ds.column_names}")
print(f"Schema:  {train_ds.features}")

## Section 2 — Inspecting rows and columns

A `Dataset` behaves like a list of dicts.  Index it to get a row,
or use a column name to get a list of values.

In [ ]:
# Row 0 is a dict of {column_name: value}
row = train_ds[0]
print(type(row))
print(f"Keys: {list(row.keys())}")
print()

# Our corpus is stored as a single string in the 'text' column
text = row["text"]
print(f"Total characters: {len(text):,}")
print(f"First 200 chars:\n\n{text[:200]}")

Because we stored the full corpus as one row, loading the text is just:

```python
text = load_dataset("transformingit/tiny-shakespeare", split="train")[0]["text"]
```

The `split=` keyword unwraps the `DatasetDict` and returns the `Dataset` directly,
saving one indexing step.

In [ ]:
# split= skips the DatasetDict layer entirely
ds = load_dataset("transformingit/tiny-shakespeare", split="train")
print(type(ds))  # Dataset, not DatasetDict
print(ds[0]["text"][:50])

## Section 3 — Caching

The first `load_dataset` call downloads and converts the data to Arrow format.
Every call after that reads from `~/.cache/huggingface/datasets/` — no network.

You can see the cache location and size:

In [ ]:
import os

from datasets import config

cache_dir = config.HF_DATASETS_CACHE
print(f"Cache location: {cache_dir}")

# Show cached datasets
if os.path.exists(cache_dir):
    entries = os.listdir(cache_dir)
    print(f"Cached datasets: {entries}")
else:
    print(
        "Cache directory not found — data may be cached elsewhere or not yet downloaded"
    )

## Section 4 — How `protoform` uses it

The `protoform.data.shakespeare` module wraps all of the above into three
clean functions so you never have to think about Hub IDs or row indexing
in your training code.

```python
from protoform.data.shakespeare import load_corpus, load_split, load_encoded
from protoform.tokenizer import CharTokenizer

# Raw text — used to build the tokenizer vocabulary
tok = CharTokenizer(load_split("train"))

# Integer tensors — used for training
splits = load_encoded(tok)
splits["train"]       # torch.Tensor, shape (1_003_854,)
splits["validation"]  # torch.Tensor, shape (55_769,)
splits["test"]        # torch.Tensor, shape (55_771,)
```

The 90/5/5 split happens inside `protoform`, not on the Hub, because
character-level models need exact byte-position boundaries — something
the `datasets` library's built-in splitting doesn't guarantee.

In [ ]:
from protoform.data.shakespeare import load_encoded, load_split
from protoform.tokenizer import CharTokenizer

tok = CharTokenizer(load_split("train"))
splits = load_encoded(tok)

for name, tensor in splits.items():
    print(f"{name:12s}  shape: {tensor.shape}  dtype: {tensor.dtype}")